# Hybrid Pipeline Test — LLM Candidate Selection + Classifier Transform Prediction

**Architecture:**
- **Stage 1 (LLM):** Phi-3-mini + LoRA selects the correct source column(s) from the schema
- **Stage 2 (Classifier):** Multiclass lightweight classifier (446K params, 119 classes) predicts the transform type
- **Comparison:** We also capture the LLM's own transform prediction to show the improvement

**Test Coverage:**
- 8 industry domains (HR, E-commerce, Healthcare, Banking, Education, Logistics, Telecom, Real Estate)
- 50+ test cases covering all major transform families
- Edge cases: noisy audit columns, ambiguous names, multi-hop joins, single-column tables, novel naming

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q torch transformers peft accelerate

## Cell 2 — Configuration

Update paths below if your files are in a different location.

In [ ]:
import os, re, time, json, math
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── Paths — update these if needed ──
LLM_ADAPTER_PATH   = "schema_matcher_llm1/adapter"       # LoRA adapter for Phi-3
CLASSIFIER_CKPT     = "multiclass_classifier_checkpoints/best.pt"  # Trained classifier
BASE_MODEL          = "microsoft/Phi-3-mini-4k-instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"\nLLM adapter  : {LLM_ADAPTER_PATH}")
print(f"Classifier   : {CLASSIFIER_CKPT}")

## Cell 3 — Load LLM (Stage 1: Candidate Selection)

In [ ]:
print("[LLM 1/3] Loading tokenizer...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ADAPTER_PATH, trust_remote_code=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print(f"[LLM 2/3] Loading base model: {BASE_MODEL}")
llm_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=False,
    attn_implementation="eager",
)

print(f"[LLM 3/3] Loading LoRA adapter...")
llm_model = PeftModel.from_pretrained(llm_model, LLM_ADAPTER_PATH)
llm_model.eval()
print(f"LLM ready on {DEVICE}  |  {sum(p.numel() for p in llm_model.parameters()):,} params")

## Cell 4 — Load Multiclass Classifier (Stage 2: Transform Prediction)

In [ ]:
# ── Reproduce the exact model architecture from train_multiclass_classifier.py ──

TYPE_VOCAB = {"<unk>": 0, "string": 1, "int": 2, "decimal": 3, "date": 4, "boolean": 5}
NUM_TYPES = len(TYPE_VOCAB)
MAX_WORD_TOKS = 24
MAX_CHAR_TOKS = 80
NUM_NUMERIC = 7

KEYWORD_GROUPS = {
    "cleaning": {"clean", "cleaned", "sanitize", "sanitized", "strip", "stripped",
                 "remove", "removed", "filter", "filtered", "purge", "purged", "wash"},
    "hashing": {"hash", "hashed", "digest", "checksum", "fingerprint", "md5", "sha"},
    "encryption": {"encrypt", "encrypted", "decrypt", "cipher", "secret", "protected", "secure"},
    "masking": {"mask", "masked", "redact", "redacted", "anon", "anonymize",
                "anonymized", "hidden", "obscure", "obscured"},
    "aggregation": {"sum", "total", "count", "avg", "average", "min", "max",
                    "aggregate", "aggregated"},
    "formatting": {"format", "formatted", "display", "pretty", "human"},
    "parsing": {"parse", "parsed", "extract", "extracted"},
    "normalization": {"normalize", "normalized", "canonical", "standard",
                      "standardized", "harmonize", "harmonized"},
    "case_change": {"lower", "upper", "initcap", "capitalize", "capitalized",
                    "lowercase", "uppercase"},
    "trimming": {"trim", "trimmed", "pad", "padded", "lpad", "rpad"},
    "splitting": {"split", "token", "segment", "part", "delimit", "delimiter"},
    "concatenation": {"concat", "concatenate", "full", "combined", "merged",
                      "complete", "join", "joined"},
    "identity": {"copy", "copied", "duplicate", "duplicated", "mirror", "same",
                 "renamed", "alias"},
    "temporal_extract": {"year", "month", "day", "hour", "minute", "quarter",
                         "week", "fiscal"},
    "temporal_general": {"date", "time", "datetime", "timestamp", "ts", "period",
                         "interval", "duration"},
    "boolean_flag": {"is", "has", "flag", "indicator", "check", "valid", "active",
                     "enabled", "disabled"},
    "bucketing": {"bucket", "bucketed", "bin", "binned", "band", "tier", "range",
                  "quantile"},
    "ranking": {"rank", "ranked", "dense", "row", "number", "position", "sequence"},
    "window_ops": {"running", "cumulative", "moving", "rolling", "lag", "lead",
                   "prev", "next", "window"},
    "lookup": {"lookup", "enrich", "enriched", "dimension", "reference", "resolved",
               "mapped"},
    "imputation": {"impute", "imputed", "fill", "filled", "default", "coalesce",
                   "fallback", "missing"},
    "json_xml": {"json", "xml", "payload", "config", "metadata", "array", "nested",
                 "struct"},
    "punctuation": {"punctuation", "punct", "symbol", "special", "character", "char"},
    "raw_marker": {"raw", "original", "input", "source", "dirty", "unprocessed",
                   "included", "with", "before"},
    "output_marker": {"output", "result", "final", "processed", "converted",
                      "transformed", "without", "after"},
    "dedup": {"dedup", "deduplicate", "deduplicated", "canonical", "unique", "distinct"},
    "slug": {"slug", "slugify", "slugified", "seo", "permalink", "url"},
    "type_conversion": {"cast", "convert", "converted", "text", "str", "num",
                        "float", "bool", "to"},
}
NUM_KW_GROUPS = len(KEYWORD_GROUPS)
KW_GROUP_KEYS = sorted(KEYWORD_GROUPS.keys())


def _name_tokens(name):
    return [w for w in re.split(r"\W+", (name or "").lower().replace("_", " ").replace("-", " ")) if w]

def _char_trigrams(word):
    w = "#" + word.lower() + "#"
    return [w[i:i+3] for i in range(len(w) - 2)]

def name_to_trigrams(name):
    tris = []
    for w in _name_tokens(name):
        tris.extend(_char_trigrams(w))
    return tris

def keyword_features(name):
    tokens = set(_name_tokens(name))
    return [1.0 if tokens & KEYWORD_GROUPS[k] else 0.0 for k in KW_GROUP_KEYS]

def type_id(t):
    return TYPE_VOCAB.get((t or "").lower().strip(), 0)


class ClassifierVocab:
    PAD_ID = 0; UNK_ID = 1
    def __init__(self, ordered_tokens):
        self._idx = {t: i for i, t in enumerate(ordered_tokens)}
    def encode(self, tokens, max_len):
        ids = [self._idx.get(t, self.UNK_ID) for t in tokens[:max_len]]
        ids += [self.PAD_ID] * (max_len - len(ids))
        return ids
    def __len__(self):
        return len(self._idx)


class MultiClassTransformPredictor(nn.Module):
    def __init__(self, word_vocab_size, tri_vocab_size, num_classes,
                 embed_dim=64, hidden=320, dropout=0.3):
        super().__init__()
        H4 = hidden // 4
        self.word_embed = nn.Embedding(word_vocab_size, embed_dim, padding_idx=0)
        self.src_word_proj = nn.Linear(embed_dim, H4)
        self.tgt_word_proj = nn.Linear(embed_dim, H4)
        self.tri_embed = nn.Embedding(tri_vocab_size, embed_dim // 2, padding_idx=0)
        self.src_tri_proj = nn.Linear(embed_dim // 2, H4)
        self.tgt_tri_proj = nn.Linear(embed_dim // 2, H4)
        self.src_type_embed = nn.Embedding(NUM_TYPES, 16)
        self.tgt_type_embed = nn.Embedding(NUM_TYPES, 16)
        self.src_kw_proj = nn.Linear(NUM_KW_GROUPS, 32)
        self.tgt_kw_proj = nn.Linear(NUM_KW_GROUPS, 32)
        self.num_proj = nn.Linear(NUM_NUMERIC, 48)
        H2 = hidden // 2
        fusion_dim = H2 + H2 + H2 + 64 + 16 + 32 + 32 + 48
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(fusion_dim), nn.Linear(fusion_dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, num_classes),
        )

    def _pool(self, embed_layer, ids):
        mask = (ids != 0).float().unsqueeze(-1)
        e = embed_layer(ids) * mask
        return e.sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, src_wids, tgt_wids, src_tids, tgt_tids,
                src_types, tgt_type, src_kw, tgt_kw, nums):
        sw = torch.relu(self.src_word_proj(self._pool(self.word_embed, src_wids)))
        tw = torch.relu(self.tgt_word_proj(self._pool(self.word_embed, tgt_wids)))
        sc = torch.relu(self.src_tri_proj(self._pool(self.tri_embed, src_tids)))
        tc = torch.relu(self.tgt_tri_proj(self._pool(self.tri_embed, tgt_tids)))
        src_rep = torch.cat([sw, sc], dim=1)
        tgt_rep = torch.cat([tw, tc], dim=1)
        interact = src_rep * tgt_rep
        src_emb = self.src_type_embed(src_types).view(src_types.size(0), -1)
        tgt_emb = self.tgt_type_embed(tgt_type)
        skw = torch.relu(self.src_kw_proj(src_kw))
        tkw = torch.relu(self.tgt_kw_proj(tgt_kw))
        num_rep = torch.relu(self.num_proj(nums))
        fused = torch.cat([src_rep, tgt_rep, interact, src_emb, tgt_emb, skw, tkw, num_rep], dim=1)
        return self.classifier(fused)


# ── Load checkpoint ──
print("Loading multiclass classifier...")
ckpt = torch.load(CLASSIFIER_CKPT, map_location=DEVICE, weights_only=False)

word_vocab = ClassifierVocab(ckpt["word_vocab_tokens"])
tri_vocab  = ClassifierVocab(ckpt["tri_vocab_tokens"])
class_names = ckpt["class_names"]
class_vocab = ckpt["class_vocab"]
temperature = ckpt.get("temperature", 1.0)

clf_model = MultiClassTransformPredictor(
    word_vocab_size=len(word_vocab), tri_vocab_size=len(tri_vocab),
    num_classes=ckpt["num_classes"], embed_dim=ckpt.get("embed_dim", 64),
    hidden=ckpt.get("hidden", 320), dropout=0.0,
)
clf_model.load_state_dict(ckpt["model_state"])
clf_model.to(DEVICE).eval()

print(f"Classifier ready  |  {ckpt['num_classes']} classes  |  "
      f"val_top1={ckpt.get('val_top1',0):.4f}  |  temperature={temperature}")
print(f"Sample classes: {class_names[:10]}...")

## Cell 5 — Hybrid Pipeline Functions

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STAGE 1: LLM — candidate selection (source columns)
# ════════════════════════════════════════════════════════════════

LLM_SYSTEM_PROMPT = """You are a schema mapping expert. Given a source database schema and a target column, identify which source column(s) should be mapped to the target and what transformation is needed.

Rules:
- Use ONLY columns that exist in the source schema
- Choose the MINIMUM number of source columns needed
- For surrogate keys, use the primary key of the matching entity table
- For person names combining first and last, use concat transform
- For cross-table lookups via foreign key, use fk_lookup transform
- For extracting year/month from a date, use date_part transform
- For computing differences between dates, use date_diff transform
- For math operations between columns, use arithmetic transform
- For boolean flags derived from status columns, use conditional transform
- For simple column renames or copies, use rename transform

Valid transform types: rename, direct_copy, concat, fk_lookup, date_part, date_diff, date_format, date_parse, arithmetic, conditional, bucketing, code_to_label, type_cast, lookup_join, template

Respond in EXACTLY this format:
source_columns: <table.column>, <table.column>, ...
transform_type: <transform>
reasoning: <brief explanation>"""


@torch.no_grad()
def llm_select_candidates(prompt):
    """Stage 1: Use LLM to select source columns and get its transform guess."""
    messages = [
        {"role": "system", "content": LLM_SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    try:
        text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = f"<|system|>\n{LLM_SYSTEM_PROMPT}<|end|>\n<|user|>\n{prompt}<|end|>\n<|assistant|>\n"

    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)
    out = llm_model.generate(
        **inputs, max_new_tokens=256, do_sample=False,
        pad_token_id=llm_tokenizer.pad_token_id,
    )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    response = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Parse
    result = {"source_columns": [], "llm_transform": "", "reasoning": "", "raw": response}
    m = re.search(r'source_columns?:\s*(.+?)(?:\n|$)', response, re.IGNORECASE)
    if m:
        result["source_columns"] = [c.strip() for c in m.group(1).split(",") if c.strip() and "." in c]
    m = re.search(r'transform_type:\s*(\S+)', response, re.IGNORECASE)
    if m:
        result["llm_transform"] = m.group(1).strip().lower()
    m = re.search(r'reasoning:\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    if m:
        result["reasoning"] = m.group(1).strip()[:200]
    return result


# ════════════════════════════════════════════════════════════════
#  STAGE 2: Classifier — transform type prediction
# ════════════════════════════════════════════════════════════════

@torch.no_grad()
def classify_transform(source_col_names, source_col_types, target_col_name, target_col_type):
    """
    Stage 2: Given the LLM-selected source columns and the target column,
    predict the transformation type using the multiclass classifier.

    Args:
        source_col_names: list of source column name strings (e.g. ["first_name", "last_name"])
        source_col_types: list of type strings (e.g. ["string", "string"])
        target_col_name:  target column name string (e.g. "full_name")
        target_col_type:  target type string (e.g. "string")

    Returns:
        dict with top1, top3 predictions and confidence scores
    """
    # Build features matching the training data format
    src_words, src_tris, src_kw_all = [], [], []
    for name in source_col_names:
        src_words.extend(_name_tokens(name))
        src_tris.extend(name_to_trigrams(name))
        kw = keyword_features(name)
        if not src_kw_all:
            src_kw_all = kw
        else:
            src_kw_all = [max(a, b) for a, b in zip(src_kw_all, kw)]
    if not src_kw_all:
        src_kw_all = [0.0] * NUM_KW_GROUPS

    tgt_words = _name_tokens(target_col_name)
    tgt_tris = name_to_trigrams(target_col_name)
    tgt_kw = keyword_features(target_col_name)

    # Numeric features (use defaults for entropy since we don't have actual data)
    n_src = float(len(source_col_names))
    pk = 0.0
    overlap = float(len(set(src_words) & set(tgt_words)))
    nums = [2.0, 2.0, 2.0, n_src, pk, 0.0, overlap]  # reasonable defaults

    # Encode
    src_wids = torch.tensor([word_vocab.encode(src_words, MAX_WORD_TOKS)], dtype=torch.long).to(DEVICE)
    tgt_wids = torch.tensor([word_vocab.encode(tgt_words, MAX_WORD_TOKS)], dtype=torch.long).to(DEVICE)
    src_tids = torch.tensor([tri_vocab.encode(src_tris, MAX_CHAR_TOKS)], dtype=torch.long).to(DEVICE)
    tgt_tids = torch.tensor([tri_vocab.encode(tgt_tris, MAX_CHAR_TOKS)], dtype=torch.long).to(DEVICE)

    src_type_ids = [type_id(t) for t in source_col_types[:4]]
    src_type_ids += [0] * (4 - len(src_type_ids))
    src_types = torch.tensor([src_type_ids], dtype=torch.long).to(DEVICE)
    tgt_type = torch.tensor([type_id(target_col_type)], dtype=torch.long).to(DEVICE)

    src_kw_t = torch.tensor([src_kw_all], dtype=torch.float32).to(DEVICE)
    tgt_kw_t = torch.tensor([tgt_kw], dtype=torch.float32).to(DEVICE)
    nums_t = torch.tensor([nums], dtype=torch.float32).to(DEVICE)

    # Forward pass
    logits = clf_model(src_wids, tgt_wids, src_tids, tgt_tids,
                       src_types, tgt_type, src_kw_t, tgt_kw_t, nums_t)

    # Apply temperature scaling
    probs = torch.softmax(logits / temperature, dim=-1)
    top3_probs, top3_ids = probs.topk(3, dim=-1)

    top3 = []
    for i in range(3):
        idx = top3_ids[0, i].item()
        top3.append((class_names[idx], round(top3_probs[0, i].item(), 4)))

    return {
        "top1": top3[0][0],
        "top1_conf": top3[0][1],
        "top3": top3,
    }


# ════════════════════════════════════════════════════════════════
#  HYBRID: Combined pipeline
# ════════════════════════════════════════════════════════════════

def hybrid_predict(schema_text, target_str, source_col_types_map=None):
    """
    Full hybrid pipeline:
      1. LLM selects source columns
      2. Classifier predicts transform type

    Args:
        schema_text: the full schema string
        target_str: target column description
        source_col_types_map: dict mapping "table.column" -> type string
    Returns:
        dict with source_columns, classifier_transform, llm_transform, etc.
    """
    prompt = f"{schema_text}\n\nMap target: {target_str}"

    # Stage 1: LLM
    t0 = time.time()
    llm_result = llm_select_candidates(prompt)
    llm_time = time.time() - t0

    # Extract column names and types for classifier
    src_names = []
    src_types = []
    for col_ref in llm_result["source_columns"]:
        parts = col_ref.split(".", 1)
        if len(parts) == 2:
            col_name = parts[1]  # just the column name part
            src_names.append(col_name)
            if source_col_types_map and col_ref.lower() in source_col_types_map:
                src_types.append(source_col_types_map[col_ref.lower()])
            else:
                src_types.append("string")  # default

    # Parse target info
    tgt_name = target_str.split("(")[0].strip().split(".")[-1].strip()
    tgt_type_m = re.search(r'\((\w+)\)', target_str)
    tgt_type = tgt_type_m.group(1) if tgt_type_m else "string"

    # Stage 2: Classifier
    t1 = time.time()
    if src_names:
        clf_result = classify_transform(src_names, src_types, tgt_name, tgt_type)
    else:
        clf_result = {"top1": "unmapped", "top1_conf": 0.0, "top3": []}
    clf_time = time.time() - t1

    return {
        "source_columns": llm_result["source_columns"],
        "llm_transform": llm_result["llm_transform"],
        "classifier_transform": clf_result["top1"],
        "classifier_confidence": clf_result["top1_conf"],
        "classifier_top3": clf_result["top3"],
        "reasoning": llm_result["reasoning"],
        "llm_time": round(llm_time, 3),
        "clf_time": round(clf_time, 5),
    }

print("Hybrid pipeline functions ready.")

## Cell 6 — Define Comprehensive Test Suite (50+ cases, 8 domains)

In [ ]:
TESTS = [
    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 1: HR / Employee Management
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "HR Schema",
        "schema": (
            "Source Schema:\n"
            "  employees(emp_id PK int, first_name string, last_name string, "
            "email string, hire_date date, salary decimal, department_id FK int)\n"
            "  departments(dept_id PK int, dept_name string, location string, budget decimal)\n"
            "\nJoins:\n  employees.department_id = departments.dept_id"
        ),
        "col_types": {
            "employees.emp_id": "int", "employees.first_name": "string",
            "employees.last_name": "string", "employees.email": "string",
            "employees.hire_date": "date", "employees.salary": "decimal",
            "employees.department_id": "int", "departments.dept_id": "int",
            "departments.dept_name": "string", "departments.location": "string",
            "departments.budget": "decimal",
        },
        "targets": [
            ('dim_employee.employee_key (int) - "surrogate key"',
             ["employees.emp_id"], "rename_only"),
            ('dim_employee.full_name (string) - "employee full name"',
             ["employees.first_name", "employees.last_name"], "concat"),
            ('dim_employee.hire_year (int) - "year hired"',
             ["employees.hire_date"], "extract_year"),
            ('dim_employee.department_name (string) - "department name via FK"',
             ["departments.dept_name"], "fk_dimension_enrichment"),
            ('dim_employee.annual_salary (decimal) - "yearly salary"',
             ["employees.salary"], "rename_only"),
            ('dim_employee.email_lower (string) - "email in lowercase"',
             ["employees.email"], "lower"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 2: E-commerce / Orders
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "E-commerce Schema",
        "schema": (
            "Source Schema:\n"
            "  orders(order_id PK int, customer_id FK int, order_date date, "
            "total_amount decimal, status string, discount_pct decimal)\n"
            "  customers(customer_id PK int, first_name string, last_name string, "
            "email string, country string, phone string)\n"
            "  order_items(item_id PK int, order_id FK int, product_id FK int, "
            "quantity int, unit_price decimal)\n"
            "  products(product_id PK int, product_name string, category_code string, "
            "weight_grams int, sku string)\n"
            "\nJoins:\n"
            "  orders.customer_id = customers.customer_id\n"
            "  order_items.order_id = orders.order_id\n"
            "  order_items.product_id = products.product_id"
        ),
        "col_types": {
            "orders.order_id": "int", "orders.customer_id": "int",
            "orders.order_date": "date", "orders.total_amount": "decimal",
            "orders.status": "string", "orders.discount_pct": "decimal",
            "customers.customer_id": "int", "customers.first_name": "string",
            "customers.last_name": "string", "customers.email": "string",
            "customers.country": "string", "customers.phone": "string",
            "order_items.item_id": "int", "order_items.order_id": "int",
            "order_items.product_id": "int", "order_items.quantity": "int",
            "order_items.unit_price": "decimal",
            "products.product_id": "int", "products.product_name": "string",
            "products.category_code": "string", "products.weight_grams": "int",
            "products.sku": "string",
        },
        "targets": [
            ('fact_order.customer_name (string) - "full name of customer"',
             ["customers.first_name", "customers.last_name"], "concat"),
            ('fact_order.order_year (int) - "year the order was placed"',
             ["orders.order_date"], "extract_year"),
            ('fact_order.is_completed (boolean) - "whether order is complete"',
             ["orders.status"], "threshold_flag"),
            ('fact_sales.line_total (decimal) - "quantity times unit price"',
             ["order_items.quantity", "order_items.unit_price"], "multiply"),
            ('dim_product.category_label (string) - "human readable category"',
             ["products.category_code"], "code_to_label"),
            ('dim_product.product_name_upper (string) - "product name in uppercase"',
             ["products.product_name"], "upper"),
            ('dim_customer.phone_masked (string) - "masked phone number"',
             ["customers.phone"], "masking"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 3: Healthcare
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Healthcare Schema",
        "schema": (
            "Source Schema:\n"
            "  patients(patient_id PK int, first_name string, last_name string, "
            "dob date, gender_code string, insurance_id FK int, ssn string)\n"
            "  insurance_plans(plan_id PK int, plan_name string, provider_name string, "
            "coverage_type_code string)\n"
            "  visits(visit_id PK int, patient_id FK int, doctor_id FK int, "
            "visit_date date, diagnosis_code string, visit_cost decimal)\n"
            "  doctors(doctor_id PK int, first_name string, last_name string, "
            "specialization_code string, department_id FK int)\n"
            "  hospital_departments(dept_id PK int, dept_name string, floor int)\n"
            "\nJoins:\n"
            "  patients.insurance_id = insurance_plans.plan_id\n"
            "  visits.patient_id = patients.patient_id\n"
            "  visits.doctor_id = doctors.doctor_id\n"
            "  doctors.department_id = hospital_departments.dept_id"
        ),
        "col_types": {
            "patients.patient_id": "int", "patients.first_name": "string",
            "patients.last_name": "string", "patients.dob": "date",
            "patients.gender_code": "string", "patients.insurance_id": "int",
            "patients.ssn": "string",
            "insurance_plans.plan_id": "int", "insurance_plans.plan_name": "string",
            "visits.visit_id": "int", "visits.patient_id": "int",
            "visits.doctor_id": "int", "visits.visit_date": "date",
            "visits.diagnosis_code": "string", "visits.visit_cost": "decimal",
            "doctors.doctor_id": "int", "doctors.first_name": "string",
            "doctors.last_name": "string", "doctors.specialization_code": "string",
            "hospital_departments.dept_id": "int", "hospital_departments.dept_name": "string",
        },
        "targets": [
            ('dim_patient.patient_name (string) - "full name of patient"',
             ["patients.first_name", "patients.last_name"], "concat"),
            ('dim_patient.birth_year (int) - "year patient was born"',
             ["patients.dob"], "extract_year"),
            ('dim_patient.gender_label (string) - "gender in readable form"',
             ["patients.gender_code"], "code_to_label"),
            ('dim_patient.insurance_plan (string) - "name of patient insurance plan"',
             ["insurance_plans.plan_name"], "fk_dimension_enrichment"),
            ('fact_visit.patient_age_at_visit (int) - "patient age at time of visit"',
             ["patients.dob", "visits.visit_date"], "date_difference"),
            ('dim_patient.ssn_redacted (string) - "redacted SSN for compliance"',
             ["patients.ssn"], "redaction"),
            ('fact_visit.visit_cost_rounded (decimal) - "cost rounded to nearest dollar"',
             ["visits.visit_cost"], "round"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 4: Banking (with noisy audit columns)
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Banking Schema (noisy)",
        "schema": (
            "Source Schema:\n"
            "  bank_accounts(account_id PK int, account_number string, "
            "account_type_code string, customer_id FK int, open_date date, "
            "balance decimal, status_code string, created_by string, updated_at datetime)\n"
            "  bank_customers(customer_id PK int, first_name string, last_name string, "
            "email string, credit_score int, income_annual decimal, "
            "risk_category_code string, created_by string, updated_at datetime)\n"
            "  bank_transactions(txn_id PK int, account_id FK int, txn_date date, "
            "amount decimal, txn_type_code string, description string, created_by string)\n"
            "\nJoins:\n"
            "  bank_accounts.customer_id = bank_customers.customer_id\n"
            "  bank_transactions.account_id = bank_accounts.account_id"
        ),
        "col_types": {
            "bank_accounts.account_id": "int", "bank_accounts.account_number": "string",
            "bank_accounts.account_type_code": "string", "bank_accounts.customer_id": "int",
            "bank_accounts.open_date": "date", "bank_accounts.balance": "decimal",
            "bank_accounts.status_code": "string",
            "bank_customers.customer_id": "int", "bank_customers.first_name": "string",
            "bank_customers.last_name": "string", "bank_customers.email": "string",
            "bank_customers.credit_score": "int", "bank_customers.income_annual": "decimal",
            "bank_customers.risk_category_code": "string",
            "bank_transactions.txn_id": "int", "bank_transactions.account_id": "int",
            "bank_transactions.txn_date": "date", "bank_transactions.amount": "decimal",
            "bank_transactions.txn_type_code": "string", "bank_transactions.description": "string",
        },
        "targets": [
            ('dim_customer.customer_name (string) - "customer full name"',
             ["bank_customers.first_name", "bank_customers.last_name"], "concat"),
            ('dim_account.account_type_label (string) - "human readable account type"',
             ["bank_accounts.account_type_code"], "code_to_label"),
            ('dim_account.is_active (boolean) - "whether account is currently open"',
             ["bank_accounts.status_code"], "threshold_flag"),
            ('dim_customer.income_bracket (string) - "income range bucket"',
             ["bank_customers.income_annual"], "bucketing_binning"),
            ('fact_txn.transaction_year (int) - "year of transaction"',
             ["bank_transactions.txn_date"], "extract_year"),
            ('dim_customer.email_hashed (string) - "hashed email for privacy"',
             ["bank_customers.email"], "hash"),
            ('fact_txn.txn_amount_abs (decimal) - "absolute transaction amount"',
             ["bank_transactions.amount"], "abs"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 5: Education
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Education Schema",
        "schema": (
            "Source Schema:\n"
            "  students(student_id PK int, first_name string, last_name string, "
            "enrollment_date date, gpa decimal, major_code string, advisor_id FK int)\n"
            "  professors(prof_id PK int, first_name string, last_name string, "
            "department string, title string)\n"
            "  courses(course_id PK int, course_name string, credits int, dept_code string)\n"
            "  enrollments(enroll_id PK int, student_id FK int, course_id FK int, "
            "grade string, semester string)\n"
            "\nJoins:\n"
            "  students.advisor_id = professors.prof_id\n"
            "  enrollments.student_id = students.student_id\n"
            "  enrollments.course_id = courses.course_id"
        ),
        "col_types": {
            "students.student_id": "int", "students.first_name": "string",
            "students.last_name": "string", "students.enrollment_date": "date",
            "students.gpa": "decimal", "students.major_code": "string",
            "professors.prof_id": "int", "professors.first_name": "string",
            "professors.last_name": "string", "professors.department": "string",
            "courses.course_id": "int", "courses.course_name": "string",
            "courses.credits": "int", "courses.dept_code": "string",
            "enrollments.enroll_id": "int", "enrollments.student_id": "int",
            "enrollments.course_id": "int", "enrollments.grade": "string",
            "enrollments.semester": "string",
        },
        "targets": [
            ('dim_student.student_name (string) - "full name of student"',
             ["students.first_name", "students.last_name"], "concat"),
            ('dim_student.enrollment_year (int) - "year student enrolled"',
             ["students.enrollment_date"], "extract_year"),
            ('dim_student.gpa_rounded (decimal) - "GPA rounded to 1 decimal"',
             ["students.gpa"], "round"),
            ('dim_student.major_label (string) - "human readable major name"',
             ["students.major_code"], "code_to_label"),
            ('dim_student.advisor_name (string) - "full name of advisor"',
             ["professors.first_name", "professors.last_name"], "concat"),
            ('dim_student.student_id_copy (int) - "direct copy of student ID"',
             ["students.student_id"], "direct_copy"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 6: Logistics / Shipping
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Logistics Schema",
        "schema": (
            "Source Schema:\n"
            "  shipments(shipment_id PK int, origin_warehouse_id FK int, "
            "dest_address string, ship_date date, delivery_date date, "
            "weight_kg decimal, cost decimal, status_code string, tracking_number string)\n"
            "  warehouses(warehouse_id PK int, warehouse_name string, city string, "
            "country string, capacity_sqft int)\n"
            "\nJoins:\n  shipments.origin_warehouse_id = warehouses.warehouse_id"
        ),
        "col_types": {
            "shipments.shipment_id": "int", "shipments.origin_warehouse_id": "int",
            "shipments.dest_address": "string", "shipments.ship_date": "date",
            "shipments.delivery_date": "date", "shipments.weight_kg": "decimal",
            "shipments.cost": "decimal", "shipments.status_code": "string",
            "shipments.tracking_number": "string",
            "warehouses.warehouse_id": "int", "warehouses.warehouse_name": "string",
            "warehouses.city": "string", "warehouses.country": "string",
        },
        "targets": [
            ('fact_shipment.transit_days (int) - "days between ship and delivery"',
             ["shipments.ship_date", "shipments.delivery_date"], "date_difference"),
            ('fact_shipment.ship_year (int) - "year of shipment"',
             ["shipments.ship_date"], "extract_year"),
            ('fact_shipment.is_delivered (boolean) - "whether shipment was delivered"',
             ["shipments.status_code"], "threshold_flag"),
            ('fact_shipment.weight_lbs (decimal) - "weight converted to pounds"',
             ["shipments.weight_kg"], "scaling_unit_conversion"),
            ('dim_warehouse.warehouse_name (string) - "origin warehouse name via FK"',
             ["warehouses.warehouse_name"], "fk_dimension_enrichment"),
            ('fact_shipment.cost_per_kg (decimal) - "shipping cost divided by weight"',
             ["shipments.cost", "shipments.weight_kg"], "divide"),
            ('fact_shipment.dest_address_trimmed (string) - "trimmed destination address"',
             ["shipments.dest_address"], "trim"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 7: Telecom
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Telecom Schema",
        "schema": (
            "Source Schema:\n"
            "  subscribers(sub_id PK int, first_name string, last_name string, "
            "phone_number string, plan_code string, activation_date date, "
            "monthly_charge decimal, data_usage_gb decimal)\n"
            "  call_records(call_id PK int, sub_id FK int, call_date date, "
            "duration_seconds int, call_type_code string, destination_number string)\n"
            "\nJoins:\n  call_records.sub_id = subscribers.sub_id"
        ),
        "col_types": {
            "subscribers.sub_id": "int", "subscribers.first_name": "string",
            "subscribers.last_name": "string", "subscribers.phone_number": "string",
            "subscribers.plan_code": "string", "subscribers.activation_date": "date",
            "subscribers.monthly_charge": "decimal", "subscribers.data_usage_gb": "decimal",
            "call_records.call_id": "int", "call_records.sub_id": "int",
            "call_records.call_date": "date", "call_records.duration_seconds": "int",
            "call_records.call_type_code": "string", "call_records.destination_number": "string",
        },
        "targets": [
            ('dim_subscriber.subscriber_name (string) - "full name"',
             ["subscribers.first_name", "subscribers.last_name"], "concat"),
            ('dim_subscriber.plan_label (string) - "human readable plan name"',
             ["subscribers.plan_code"], "code_to_label"),
            ('dim_subscriber.activation_year (int) - "year activated"',
             ["subscribers.activation_date"], "extract_year"),
            ('fact_call.duration_minutes (decimal) - "call duration in minutes"',
             ["call_records.duration_seconds"], "scaling_unit_conversion"),
            ('fact_call.call_type_label (string) - "readable call type"',
             ["call_records.call_type_code"], "code_to_label"),
            ('dim_subscriber.phone_anonymized (string) - "anonymized phone"',
             ["subscribers.phone_number"], "anonymization"),
        ],
    },

    # ═══════════════════════════════════════════════════════════════
    #  SCENARIO 8: Real Estate
    # ═══════════════════════════════════════════════════════════════
    {
        "name": "Real Estate Schema",
        "schema": (
            "Source Schema:\n"
            "  properties(property_id PK int, address string, city string, "
            "zip_code string, listing_price decimal, sqft int, bedrooms int, "
            "listing_date date, status_code string, agent_id FK int)\n"
            "  agents(agent_id PK int, first_name string, last_name string, "
            "license_number string, email string, commission_rate decimal)\n"
            "  transactions(txn_id PK int, property_id FK int, buyer_name string, "
            "sale_price decimal, sale_date date, closing_date date)\n"
            "\nJoins:\n"
            "  properties.agent_id = agents.agent_id\n"
            "  transactions.property_id = properties.property_id"
        ),
        "col_types": {
            "properties.property_id": "int", "properties.address": "string",
            "properties.city": "string", "properties.zip_code": "string",
            "properties.listing_price": "decimal", "properties.sqft": "int",
            "properties.bedrooms": "int", "properties.listing_date": "date",
            "properties.status_code": "string", "properties.agent_id": "int",
            "agents.agent_id": "int", "agents.first_name": "string",
            "agents.last_name": "string", "agents.license_number": "string",
            "agents.email": "string", "agents.commission_rate": "decimal",
            "transactions.txn_id": "int", "transactions.property_id": "int",
            "transactions.buyer_name": "string", "transactions.sale_price": "decimal",
            "transactions.sale_date": "date", "transactions.closing_date": "date",
        },
        "targets": [
            ('dim_property.price_per_sqft (decimal) - "listing price divided by square footage"',
             ["properties.listing_price", "properties.sqft"], "divide"),
            ('dim_property.listing_year (int) - "year property was listed"',
             ["properties.listing_date"], "extract_year"),
            ('dim_property.is_sold (boolean) - "whether property is sold"',
             ["properties.status_code"], "threshold_flag"),
            ('dim_property.bedroom_tier (string) - "bedroom count bucket"',
             ["properties.bedrooms"], "bucketing_binning"),
            ('dim_agent.agent_name (string) - "full name of listing agent"',
             ["agents.first_name", "agents.last_name"], "concat"),
            ('fact_sale.days_on_market (int) - "days between listing and sale"',
             ["properties.listing_date", "transactions.sale_date"], "date_difference"),
            ('fact_sale.commission_amount (decimal) - "sale price times commission rate"',
             ["transactions.sale_price", "agents.commission_rate"], "multiply"),
            ('dim_property.address_initcap (string) - "address in title case"',
             ["properties.address"], "initcap"),
            ('dim_property.sqft_cast (decimal) - "sqft cast to decimal"',
             ["properties.sqft"], "type_cast"),
        ],
    },
]

total_tests = sum(len(t["targets"]) for t in TESTS)
print(f"Loaded {total_tests} test cases across {len(TESTS)} schemas.")
for s in TESTS:
    print(f"  {s['name']}: {len(s['targets'])} cases")

## Cell 7 — Run All Tests

In [ ]:
# ── Accumulators ──
total = 0
col_correct = 0
llm_tf_correct = 0
clf_tf_correct = 0
hybrid_both_correct = 0
llm_both_correct = 0
total_llm_time = 0.0
total_clf_time = 0.0
all_results = []
failures = []

print("=" * 90)
print(f"  HYBRID PIPELINE TEST  —  {total_tests} cases across {len(TESTS)} schemas")
print(f"  Stage 1: LLM (candidate selection)  |  Stage 2: Classifier (transform prediction)")
print("=" * 90)

for scenario in TESTS:
    print(f"\n{'─'*70}")
    print(f"  {scenario['name']}")
    print(f"{'─'*70}")

    for target_str, exp_cols, exp_tf in scenario["targets"]:
        total += 1

        result = hybrid_predict(
            scenario["schema"], target_str,
            source_col_types_map=scenario.get("col_types", {}),
        )

        pred_cols = sorted([c.lower().strip() for c in result["source_columns"]])
        expected  = sorted([c.lower().strip() for c in exp_cols])
        pred_llm_tf = result["llm_transform"]
        pred_clf_tf = result["classifier_transform"]

        cols_ok    = pred_cols == expected
        llm_tf_ok  = pred_llm_tf == exp_tf
        clf_tf_ok  = pred_clf_tf == exp_tf

        if cols_ok:    col_correct += 1
        if llm_tf_ok:  llm_tf_correct += 1
        if clf_tf_ok:  clf_tf_correct += 1
        if cols_ok and clf_tf_ok: hybrid_both_correct += 1
        if cols_ok and llm_tf_ok: llm_both_correct += 1
        total_llm_time += result["llm_time"]
        total_clf_time += result["clf_time"]

        status = "PASS" if (cols_ok and clf_tf_ok) else "FAIL"
        target_short = target_str.split(" - ")[0] if " - " in target_str else target_str[:50]

        # Per-case output
        print(f"  [{status}] {target_short}")
        print(f"         Cols       : {pred_cols}  {'OK' if cols_ok else 'EXPECTED ' + str(expected)}")
        print(f"         LLM tf     : {pred_llm_tf}  {'OK' if llm_tf_ok else 'WRONG (expected ' + exp_tf + ')'}")
        print(f"         Clf tf     : {pred_clf_tf} ({result['classifier_confidence']:.2%})  "
              f"{'OK' if clf_tf_ok else 'WRONG (expected ' + exp_tf + ')'}")
        if result.get("classifier_top3"):
            top3_str = ", ".join(f"{n}({p:.1%})" for n, p in result["classifier_top3"])
            print(f"         Clf top3   : {top3_str}")
        print(f"         Time       : LLM={result['llm_time']:.2f}s  Clf={result['clf_time']*1000:.1f}ms")

        record = {
            "scenario": scenario["name"], "target": target_short,
            "expected_cols": expected, "predicted_cols": pred_cols, "cols_correct": cols_ok,
            "expected_tf": exp_tf,
            "llm_tf": pred_llm_tf, "llm_tf_correct": llm_tf_ok,
            "clf_tf": pred_clf_tf, "clf_tf_correct": clf_tf_ok,
            "clf_confidence": result["classifier_confidence"],
            "clf_top3": result["classifier_top3"],
            "hybrid_pass": cols_ok and clf_tf_ok,
            "llm_pass": cols_ok and llm_tf_ok,
            "reasoning": result.get("reasoning", ""),
            "llm_time": result["llm_time"],
            "clf_time": result["clf_time"],
        }
        all_results.append(record)
        if not (cols_ok and clf_tf_ok):
            failures.append(record)

print("\n" + "=" * 90)
print("  All tests complete.")
print("=" * 90)

## Cell 8 — Results Summary: LLM-Only vs Hybrid

In [ ]:
n = total

print("=" * 90)
print("  HEAD-TO-HEAD COMPARISON: LLM-Only vs Hybrid Pipeline")
print("=" * 90)
print(f"")
print(f"  {'Metric':<35} {'LLM-Only':>12} {'Hybrid':>12} {'Delta':>10}")
print(f"  {'─'*35} {'─'*12} {'─'*12} {'─'*10}")
print(f"  {'Source column accuracy':<35} {col_correct/n:>11.1%} {col_correct/n:>11.1%} {'same':>10}")
print(f"  {'Transform accuracy':<35} {llm_tf_correct/n:>11.1%} {clf_tf_correct/n:>11.1%} "
      f"{'+' if clf_tf_correct > llm_tf_correct else ''}{(clf_tf_correct-llm_tf_correct)/n:>9.1%}")
print(f"  {'Both correct (cols + transform)':<35} {llm_both_correct/n:>11.1%} {hybrid_both_correct/n:>11.1%} "
      f"{'+' if hybrid_both_correct > llm_both_correct else ''}{(hybrid_both_correct-llm_both_correct)/n:>9.1%}")
print(f"")
print(f"  {'Avg LLM time / query':<35} {total_llm_time/n:>11.2f}s")
print(f"  {'Avg Classifier time / query':<35} {total_clf_time/n*1000:>10.1f}ms")
print(f"  {'Total time':<35} {total_llm_time + total_clf_time:>11.1f}s")
print("=" * 90)

# ── Per-scenario breakdown ──
print(f"\n  Per-Scenario Breakdown:")
print(f"  {'Scenario':<35} {'Cols':>6} {'LLM tf':>8} {'Clf tf':>8} {'Hybrid':>8} {'LLM-only':>10}")
print(f"  {'─'*35} {'─'*6} {'─'*8} {'─'*8} {'─'*8} {'─'*10}")
for scenario in TESTS:
    sc = [r for r in all_results if r["scenario"] == scenario["name"]]
    sc_n = len(sc)
    sc_cols = sum(1 for r in sc if r["cols_correct"])
    sc_llm  = sum(1 for r in sc if r["llm_tf_correct"])
    sc_clf  = sum(1 for r in sc if r["clf_tf_correct"])
    sc_hyb  = sum(1 for r in sc if r["hybrid_pass"])
    sc_lonly = sum(1 for r in sc if r["llm_pass"])
    print(f"  {scenario['name']:<35} {sc_cols}/{sc_n:>3}  {sc_llm}/{sc_n:>5}  "
          f"{sc_clf}/{sc_n:>5}  {sc_hyb}/{sc_n:>5}  {sc_lonly}/{sc_n:>7}")

# ── Per-transform breakdown ──
print(f"\n  Per-Transform Breakdown (Classifier):")
tf_types = sorted(set(r["expected_tf"] for r in all_results))
print(f"  {'Transform':<30} {'N':>4} {'Cols':>6} {'Clf OK':>8} {'LLM OK':>8}")
print(f"  {'─'*30} {'─'*4} {'─'*6} {'─'*8} {'─'*8}")
for tf in tf_types:
    tf_res = [r for r in all_results if r["expected_tf"] == tf]
    tf_n = len(tf_res)
    c_ok = sum(1 for r in tf_res if r["cols_correct"])
    clf_ok = sum(1 for r in tf_res if r["clf_tf_correct"])
    llm_ok = sum(1 for r in tf_res if r["llm_tf_correct"])
    print(f"  {tf:<30} {tf_n:>4} {c_ok:>6} {clf_ok:>8} {llm_ok:>8}")

## Cell 9 — Failure Analysis

In [ ]:
if failures:
    print(f"{'='*90}")
    print(f"  FAILURES — {len(failures)} / {n}")
    print(f"{'='*90}")

    # Categorize failures
    col_failures = [f for f in failures if not f["cols_correct"]]
    tf_failures  = [f for f in failures if f["cols_correct"] and not f["clf_tf_correct"]]

    if col_failures:
        print(f"\n  --- Column Selection Failures ({len(col_failures)}) ---")
        for i, f in enumerate(col_failures, 1):
            print(f"  [{i}] {f['scenario']}  >  {f['target']}")
            print(f"      Predicted: {f['predicted_cols']}")
            print(f"      Expected : {f['expected_cols']}")

    if tf_failures:
        print(f"\n  --- Transform Failures (cols correct, transform wrong) ({len(tf_failures)}) ---")
        for i, f in enumerate(tf_failures, 1):
            print(f"  [{i}] {f['scenario']}  >  {f['target']}")
            print(f"      Clf predicted : {f['clf_tf']} ({f['clf_confidence']:.1%})")
            print(f"      Expected      : {f['expected_tf']}")
            print(f"      Clf top3      : {f['clf_top3']}")
            print(f"      LLM predicted : {f['llm_tf']}")

    # Cases where hybrid fixed LLM failures
    fixed = [r for r in all_results if r["hybrid_pass"] and not r["llm_pass"]]
    if fixed:
        print(f"\n  --- Cases FIXED by Hybrid (LLM wrong, Classifier correct) ({len(fixed)}) ---")
        for i, f in enumerate(fixed, 1):
            print(f"  [{i}] {f['scenario']}  >  {f['target']}")
            print(f"      LLM said: {f['llm_tf']}  |  Clf said: {f['clf_tf']}  |  Expected: {f['expected_tf']}")

    # Cases where hybrid broke what LLM got right
    broken = [r for r in all_results if r["llm_pass"] and not r["hybrid_pass"]]
    if broken:
        print(f"\n  --- Cases BROKEN by Hybrid (LLM right, Classifier wrong) ({len(broken)}) ---")
        for i, f in enumerate(broken, 1):
            print(f"  [{i}] {f['scenario']}  >  {f['target']}")
            print(f"      LLM said: {f['llm_tf']}  |  Clf said: {f['clf_tf']}  |  Expected: {f['expected_tf']}")
else:
    print("\n  ALL TESTS PASSED — No failures!")

## Cell 10 — Export Results to JSON

In [ ]:
export = {
    "summary": {
        "total_tests":           n,
        "col_accuracy":          round(col_correct / n, 4),
        "llm_transform_acc":     round(llm_tf_correct / n, 4),
        "clf_transform_acc":     round(clf_tf_correct / n, 4),
        "llm_only_both_acc":     round(llm_both_correct / n, 4),
        "hybrid_both_acc":       round(hybrid_both_correct / n, 4),
        "improvement":           round((hybrid_both_correct - llm_both_correct) / n, 4),
        "avg_llm_time":          round(total_llm_time / n, 3),
        "avg_clf_time_ms":       round(total_clf_time / n * 1000, 3),
        "device":                DEVICE,
        "gpu":                   torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A",
    },
    "results": all_results,
}

out_path = "hybrid_pipeline_test_results.json"
with open(out_path, "w") as fp:
    json.dump(export, fp, indent=2, default=str)

print(f"Results exported to: {out_path}")
print(json.dumps(export["summary"], indent=2))

## Cell 11 — Interactive: Test Any Custom Schema

In [ ]:
# ── Try your own mapping ──
custom_schema = """Source Schema:
  employees(emp_id PK int, first_name string, last_name string, hire_date date, salary decimal, dept_code string)
  departments(dept_code PK string, dept_name string, manager_id FK int)

Joins:
  employees.dept_code = departments.dept_code"""

custom_target = 'dim_employee.dept_name_upper (string) - "department name in uppercase via lookup"'

custom_types = {
    "employees.emp_id": "int", "employees.first_name": "string",
    "employees.last_name": "string", "employees.hire_date": "date",
    "employees.salary": "decimal", "employees.dept_code": "string",
    "departments.dept_code": "string", "departments.dept_name": "string",
}

result = hybrid_predict(custom_schema, custom_target, custom_types)
print(f"Source columns     : {result['source_columns']}")
print(f"LLM transform      : {result['llm_transform']}")
print(f"Classifier transform: {result['classifier_transform']} ({result['classifier_confidence']:.1%})")
print(f"Classifier top-3   : {result['classifier_top3']}")
print(f"Reasoning          : {result['reasoning'][:200]}")
print(f"Time               : LLM={result['llm_time']:.2f}s  Clf={result['clf_time']*1000:.1f}ms")